# 01 — Inspección Inicial

**Proyecto Integrador — Minería de Datos 1**
**Dataset:** `streaming_users_dirty.json` — usuarios de una plataforma de streaming.

**Objetivo de esta etapa:** comprender la estructura general del dataset, sus tipos de
datos, dimensiones, valores faltantes y duplicados, y reunir evidencia para orientar las
decisiones de la etapa de calidad y limpieza. En esta etapa **no se toman decisiones
definitivas de limpieza**, solo se documenta lo observado.


## 0. Entorno de ejecución (Google Colab / local)

Esta celda detecta si el notebook corre en **Google Colab**. Si es así, monta Google
Drive y define `BASE_PATH` apuntando a la carpeta del proyecto dentro de tu Drive. Si
corre localmente (Jupyter/VS Code), usa la ruta relativa habitual del repositorio.

> **Antes de ejecutar en Colab:** subí la carpeta completa `PI_Mineria_Datos_1/`
> (con su estructura de subcarpetas `data/`, `notebooks/`, `logs/`, etc.) a tu Google
> Drive, por ejemplo en `MyDrive/PI_Mineria_Datos_1/`. Si la ubicaste en otro lugar,
> ajustá la variable `BASE_PATH` de la celda de abajo.


In [1]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Ruta del proyecto dentro de Google Drive (ajustar si corresponde)
BASE_PATH = "/content/drive/MyDrive/PI_Mineria_Datos_1" if IN_COLAB else ".."

print(f"Ejecutando en Colab: {IN_COLAB}")
print(f"BASE_PATH: {BASE_PATH}")


Ejecutando en Colab: False
BASE_PATH: ..


In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RAW_PATH = f"{BASE_PATH}/data/raw/streaming_users_dirty.json"
df = pd.read_json(RAW_PATH)
print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")


Dimensiones del dataset: 8160 filas x 8 columnas


## 1. Estructura general y tipos de datos

In [3]:
df.dtypes.to_frame(name="tipo_de_dato")


In [4]:
df.head(10)


**Observación:** el dataset contiene 8 columnas: un identificador de usuario
(`user_id`), variables demográficas (`age`, `country`), variables de comportamiento
(`monthly_watch_time_mins`, `favorite_genre`, `last_login_date`), variables de negocio
(`subscription_plan`) y una variable de soporte (`customer_support_tickets`). Todas las
columnas de texto fueron leídas como tipo `object`/`str`, incluida `last_login_date`, que
debería representar una fecha — esto ya es un indicio a revisar en la etapa de calidad.

## 2. Valores faltantes

In [5]:
nulos = df.isnull().sum()
porcentaje = (nulos / len(df) * 100).round(2)
resumen_nulos = pd.DataFrame({"nulos": nulos, "% del total": porcentaje})
resumen_nulos[resumen_nulos["nulos"] > 0].sort_values("nulos", ascending=False)


**Observación:** existen valores faltantes explícitos (`None`/`NaN`) en tres
columnas: `last_login_date` (320, ~3.9%), `favorite_genre` (240, ~2.9%) y
`monthly_watch_time_mins` (193, ~2.4%). El resto de las columnas no presenta nulos
explícitos, aunque esto no descarta valores faltantes *implícitos* (códigos centinela,
strings vacíos o valores fuera de rango), que se investigan a continuación.

## 3. Duplicados

In [6]:
dup_filas = df.duplicated().sum()
dup_user_id = df.duplicated(subset=["user_id"]).sum()
print(f"Filas totalmente duplicadas: {dup_filas}")
print(f"user_id duplicados (misma fila puede diferir en otros campos): {dup_user_id}")


Filas totalmente duplicadas: 126
user_id duplicados (misma fila puede diferir en otros campos): 160


In [7]:
# Ejemplo de duplicados exactos
df[df.duplicated(keep=False)].sort_values("user_id").head(6)


**Observación:** hay 126 filas totalmente duplicadas (mismo `user_id` y
mismos valores en todas las columnas) y 160 registros con `user_id` repetido en total —
es decir, además de los duplicados exactos hay casos de un mismo `user_id` con valores
distintos en otras columnas, lo cual debe resolverse en la etapa de limpieza (¿error de
carga? ¿usuario con más de un registro?).

## 4. Rango y consistencia de variables numéricas

In [8]:
df[["age", "monthly_watch_time_mins", "customer_support_tickets"]].describe()


In [9]:
print("AGE")
print(f"  valores negativos: {(df['age'] < 0).sum()}")
print(f"  valores > 100 años: {(df['age'] > 100).sum()}")
print()
print("MONTHLY_WATCH_TIME_MINS")
print(f"  valores negativos: {(df['monthly_watch_time_mins'] < 0).sum()}")
print(f"  valores >= 20000 (posible código centinela): {(df['monthly_watch_time_mins'] >= 20000).sum()}")
print(f"  valor máximo observado: {df['monthly_watch_time_mins'].max()}")
print()
print("CUSTOMER_SUPPORT_TICKETS")
print(f"  valores negativos: {(df['customer_support_tickets'] < 0).sum()}")
print("  distribución de valores altos (>20):")
print(df.loc[df['customer_support_tickets'] > 20, 'customer_support_tickets'].value_counts())


AGE
  valores negativos: 21
  valores > 100 años: 53

MONTHLY_WATCH_TIME_MINS
  valores negativos: 49
  valores >= 20000 (posible código centinela): 31
  valor máximo observado: 99999.0

CUSTOMER_SUPPORT_TICKETS
  valores negativos: 29
  distribución de valores altos (>20):
customer_support_tickets
99     35
150    32
Name: count, dtype: int64


**Observación:** se detectan valores fuera de rango plausible en tres
variables numéricas:
- `age`: edades negativas (mínimo -5) y edades hasta 150 años — imposibles para un
  usuario real.
- `monthly_watch_time_mins`: minutos negativos (mínimo -120) y un valor máximo de 99999,
  que se repite de forma sospechosa y parece un **código centinela** de dato faltante o
  error de carga, no una observación real (99999 minutos = ~69 días de reproducción
  continua en un mes).
- `customer_support_tickets`: valores negativos (mínimo -1) y dos valores que se repiten
  de forma anómala (99 y 150), también compatibles con códigos centinela.

Estos hallazgos se tratarán con evidencia y justificación en `02_calidad_y_limpieza.ipynb`.


## 5. Consistencia de variables categóricas

In [10]:
for col in ["subscription_plan", "country", "favorite_genre"]:
    print(f"--- {col}: {df[col].nunique(dropna=False)} valores únicos ---")
    print(df[col].value_counts(dropna=False).to_string())
    print()


--- subscription_plan: 15 valores únicos ---
subscription_plan
Básico       3450
Estándar     2711
Premium      1519
basico         60
BASICO         52
Basic          52
básico         50
Std            48
Estándar       46
estandar       36
STANDARD       34
Premium        31
PREMIUM        26
Premiun        23
premium        22

--- country: 26 valores únicos ---
country
Brasil        1132
Chile         1132
México        1129
Uruguay       1124
Perú          1120
Colombia      1116
Argentina     1087
colombia        27
méxico          25
uruguay         24
Brazil          21
COL             19
CHL             18
URY             17
MEX             16
Chile           16
argentina       16
PER             16
chile           15
Mexico          15
Peru            15
BRA             15
brasil          13
perú            12
ARG             10
Argentina       10

--- favorite_genre: 29 valores únicos ---
favorite_genre
Comedia        1112
Drama          1105
Documental     1095
Thriller   

**Observación:** las tres variables categóricas presentan el mismo patrón
de inconsistencia: **una misma categoría real está representada con múltiples grafías**
(mayúsculas/minúsculas, con/sin tilde, abreviaturas, código ISO, y traducciones al
inglés). Por ejemplo `subscription_plan` tiene 3 categorías "limpias" (Básico, Estándar,
Premium) pero conviven variantes como "basico", "BASICO", "Basic", "Std", "STANDARD",
"Premiun", etc. Lo mismo ocurre con `country` (variantes de "Colombia", "México", código
"COL", "Brazil" en inglés) y `favorite_genre` (variantes de "Comedia", "Acción",
traducciones al inglés como "Action"/"comedy"). Este es un problema típico de **carga de
datos desde múltiples fuentes sin normalizar**, y se resolverá mapeando cada variante a
una categoría canónica en la etapa de limpieza.

## 6. Formato de fechas

In [11]:
import re

def formato_fecha(valor):
    if pd.isna(valor):
        return "nulo"
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", valor):
        return "YYYY-MM-DD"
    if re.fullmatch(r"\d{2}-\d{2}-\d{4}", valor):
        return "DD-MM-YYYY o MM-DD-YYYY"
    if re.fullmatch(r"\d{4}/\d{2}/\d{2}", valor):
        return "YYYY/MM/DD"
    return "otro/inválido"

conteo_formatos = df["last_login_date"].apply(formato_fecha).value_counts()
print(conteo_formatos)


last_login_date
YYYY-MM-DD                 7428
nulo                        320
DD-MM-YYYY o MM-DD-YYYY     265
YYYY/MM/DD                  133
otro/inválido                14
Name: count, dtype: int64


In [12]:
# Ejemplos de fechas con componentes imposibles (mes > 12 o día > 31)
def es_fecha_imposible(valor):
    if pd.isna(valor):
        return False
    partes = re.split(r"[-/]", valor)
    if len(partes) != 3:
        return True
    nums = [int(p) for p in partes]
    return any(n > 31 for n in nums[:2]) and any(n > 12 for n in nums[:2])

ejemplos_invalidos = df.loc[df["last_login_date"].apply(
    lambda v: isinstance(v, str) and (int(re.split('[-/]', v)[1]) > 12 if len(re.split('[-/]', v))==3 and re.split('[-/]', v)[0].isdigit() and len(re.split('[-/]', v)[0])==4 else False)
), "last_login_date"].unique()
print("Ejemplos con mes > 12 en formato YYYY-MM-DD:", ejemplos_invalidos[:10])


Ejemplos con mes > 12 en formato YYYY-MM-DD: <StringArray>
['2026-15-03']
Length: 1, dtype: str


**Observación:** `last_login_date` mezcla al menos tres formatos distintos
(`YYYY-MM-DD`, `DD-MM-YYYY`/`MM-DD-YYYY` ambiguo, `YYYY/MM/DD`), además de casos con
componentes imposibles (por ejemplo mes 15). Esto impide convertir la columna a fecha de
forma directa y requiere un parseo cuidadoso por formato en la etapa de limpieza, marcando
como faltante lo que no pueda resolverse sin ambigüedad.

## 7. Síntesis de hallazgos de la inspección inicial

| Hallazgo | Columnas afectadas | Evidencia |
|---|---|---|
| Nulos explícitos | `monthly_watch_time_mins`, `favorite_genre`, `last_login_date` | 193 / 240 / 320 registros |
| Duplicados | fila completa, `user_id` | 126 filas exactas; 160 `user_id` repetidos |
| Valores fuera de rango | `age`, `monthly_watch_time_mins`, `customer_support_tickets` | edades <0 o >100; minutos <0 o =99999; tickets <0 o en {99,150} |
| Categorías inconsistentes | `subscription_plan`, `country`, `favorite_genre` | múltiples grafías por categoría real |
| Formatos de fecha mixtos | `last_login_date` | 3 formatos + fechas imposibles |

Estos hallazgos, con su evidencia, orientan directamente las decisiones que se
justificarán y aplicarán en `02_calidad_y_limpieza.ipynb`.
